# 05_bea_inspection

BEA: limpeza da renda pessoal per capita. Notebook exploratório reproduzível (dados `data/raw` e `data/processed` não estão versionados). Execute com diretório de trabalho `notebooks/`. Resultados são preditivos/descritivos, **não causais**.

In [ ]:
from pathlib import Path

arquivo = Path("../data/raw/bea/personal_income.csv")

with open(arquivo, "r", encoding="utf-8-sig") as f:
    for numero, linha in enumerate(f):
        if numero >= 15:
            break
        print(f"{numero:02d}: {linha.rstrip()}")

In [ ]:
import pandas as pd

bea = pd.read_csv(
    arquivo,
    skiprows=3,
    encoding="utf-8-sig",
    dtype={"GeoFIPS": str},
    low_memory=False
)

anos = [str(ano) for ano in range(2019, 2025)]

colunas = ["GeoFIPS", "GeoName"] + anos

bea = bea[colunas].copy()

print("Dimensões:", bea.shape)
display(bea.head())

In [ ]:
bea_long = bea.melt(
    id_vars=["GeoFIPS", "GeoName"],
    value_vars=anos,
    var_name="year",
    value_name="income_per_capita"
)

bea_long = bea_long.rename(
    columns={"GeoName": "state"}
)

bea_long["state"] = (
    bea_long["state"]
    .str.replace(r"\s*\*$", "", regex=True)
    .str.strip()
)

bea_long["year"] = bea_long["year"].astype(int)

display(bea_long.head(10))
print("Dimensões:", bea_long.shape)

In [ ]:
# Recuperar os 50 estados utilizados no CDC
cdc = pd.read_csv(
    "../data/raw/cdc_homicides_2019_2024.csv"
)

estados_validos = set(cdc["name"]) - {"District of Columbia"}

# Filtrar as unidades geográficas
bea_filtrado = bea_long[
    bea_long["state"].isin(estados_validos)
].copy()

print("Dimensões:", bea_filtrado.shape)
print("Estados:", bea_filtrado["state"].nunique())

display(
    bea_filtrado.groupby("year")["state"].nunique()
)

In [ ]:
# Garantir que a renda esteja em formato numérico
bea_filtrado["income_per_capita"] = pd.to_numeric(
    bea_filtrado["income_per_capita"],
    errors="coerce"
)

print("Duplicações:",
      bea_filtrado.duplicated(["state", "year"]).sum())

print("Rendas ausentes:",
      bea_filtrado["income_per_capita"].isna().sum())

print("Rendas inválidas:",
      (bea_filtrado["income_per_capita"] <= 0).sum())

print("\nEstatísticas descritivas:")
display(bea_filtrado["income_per_capita"].describe())
assert bea_filtrado.shape[0] == 300
assert not bea_filtrado.duplicated(["state","year"]).any()
assert bea_filtrado["income_per_capita"].notna().all()


In [ ]:
from pathlib import Path

bea_final = bea_filtrado[
    ["state", "year", "income_per_capita"]
].copy()

destino = Path("../data/processed/bea_income_2019_2024.csv")

bea_final.to_csv(destino, index=False)

print("Dimensões:", bea_final.shape)
print("Salvo em:", destino.resolve())